# MedVS-AI — Chest X-ray: lung-field segmentation + normal/TB (2nd modality)

Mirrors the dermoscopy pipeline for a **second modality**, proving the platform is modality-agnostic:
- a **from-scratch Attention U-Net** segments the **lung fields**
- an **EfficientNet-B0** calls **normal vs. tuberculosis**

Data: Montgomery + Shenzhen NLM TB sets via Kaggle (`kmader/pulmonary-chest-xray-abnormalities`).

**Steps:** set a **T4 GPU** (`Runtime > Change runtime type`), then run top to bottom. You'll be asked to
paste your **Kaggle API token** (kaggle.com → Settings → *API* → *Generate New Token* → the `KGAT_…` value).
Outputs: `unet_cxr.onnx`, `classifier_cxr.onnx`, `case_bundle_cxr.zip`.
> NOT FOR CLINICAL USE.

## 1. GPU + packages

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if not torch.cuda.is_available(): print("Set Runtime > Change runtime type > T4 GPU")


In [ ]:
%pip -q install kaggle onnxruntime onnxscript scikit-learn
print("ok")


## 2. Download Montgomery + Shenzhen
Paste your Kaggle API token into the cell below (between the quotes), then run it.

In [ ]:
import os
# Paste your Kaggle API token BETWEEN THE QUOTES below, then run this cell.
# (kaggle.com -> Settings -> API -> Create New Token -> copy the KGAT_... value)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

!kaggle datasets download -d kmader/pulmonary-chest-xray-abnormalities
!mkdir -p data && unzip -q -o pulmonary-chest-xray-abnormalities.zip -d data
print("extracted:", len(os.listdir("data")), "items in data/")


## 3. Discover images + lung masks + TB labels
Recursive globbing, so the exact folder nesting doesn't matter. Filenames end `_0` (normal) or `_1` (TB).

In [ ]:
import glob, numpy as np, pandas as pd
from PIL import Image

def stem(p): return os.path.splitext(os.path.basename(p))[0]
cxr   = glob.glob("data/**/CXR_png/*.png", recursive=True)
left  = {stem(p): p for p in glob.glob("data/**/leftMask/*.png", recursive=True)}
right = {stem(p): p for p in glob.glob("data/**/rightMask/*.png", recursive=True)}

def tb_label(s):
    tok = s.split("_")[-1]
    return int(tok) if tok in ("0", "1") else None

rows = []
for p in cxr:
    s = stem(p); lab = tb_label(s)
    if lab is None: continue
    rows.append({"id": s, "image": p, "left": left.get(s), "right": right.get(s), "tb": lab})
df = pd.DataFrame(rows).drop_duplicates("id").reset_index(drop=True)
seg_df = df[df["left"].notna() & df["right"].notna()].reset_index(drop=True)
print("total CXR:", len(df), "| with lung masks (segmentation):", len(seg_df))
print("normal:", int((df.tb == 0).sum()), "| TB:", int((df.tb == 1).sum()))
assert len(df) and len(seg_df), "no data found - check the Kaggle download / folder names"


## 4. The model — same from-scratch Attention U-Net, plus CXR data helpers

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class DoubleConv(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
            nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class AttentionGate(nn.Module):
    def __init__(self, g, x, inter):
        super().__init__()
        self.tx = nn.Conv2d(x, inter, 1, bias=False); self.pg = nn.Conv2d(g, inter, 1)
        self.psi = nn.Conv2d(inter, 1, 1)
    def forward(self, g, x):
        return x * torch.sigmoid(self.psi(F.relu(self.tx(x) + self.pg(g))))

class UpBlock(nn.Module):
    def __init__(self, i, s, o):
        super().__init__()
        self.up = nn.ConvTranspose2d(i, o, 2, stride=2)
        self.att = AttentionGate(o, s, o // 2); self.conv = DoubleConv(o + s, o)
    def forward(self, x, skip):
        x = self.up(x); return self.conv(torch.cat([self.att(x, skip), x], 1))

class AttentionUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        c = [base, base*2, base*4, base*8, base*16]
        self.e1, self.e2 = DoubleConv(in_ch, c[0]), DoubleConv(c[0], c[1])
        self.e3, self.e4 = DoubleConv(c[1], c[2]), DoubleConv(c[2], c[3])
        self.pool = nn.MaxPool2d(2); self.bottleneck = DoubleConv(c[3], c[4])
        self.u4, self.u3 = UpBlock(c[4], c[3], c[3]), UpBlock(c[3], c[2], c[2])
        self.u2, self.u1 = UpBlock(c[2], c[1], c[1]), UpBlock(c[1], c[0], c[0])
        self.head = nn.Conv2d(c[0], out_ch, 1)
    def forward(self, x):
        s1 = self.e1(x); s2 = self.e2(self.pool(s1)); s3 = self.e3(self.pool(s2)); s4 = self.e4(self.pool(s3))
        b = self.bottleneck(self.pool(s4))
        x = self.u4(b, s4); x = self.u3(x, s3); x = self.u2(x, s2); x = self.u1(x, s1)
        return self.head(x)

def dice_bce_loss(logits, target, eps=1.0):
    bce = F.binary_cross_entropy_with_logits(logits, target); p = torch.sigmoid(logits)
    num = 2 * (p * target).sum((1, 2, 3)) + eps; den = (p + target).sum((1, 2, 3)) + eps
    return bce + (1 - (num / den).mean())

SIZE = 256
MEAN = np.array((0.485, 0.456, 0.406), np.float32); STD = np.array((0.229, 0.224, 0.225), np.float32)
def load_img(p):
    im = Image.open(p).convert("RGB").resize((SIZE, SIZE))
    return ((np.asarray(im, np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1)
def lung_mask(r):
    l = np.asarray(Image.open(r["left"]).convert("L").resize((SIZE, SIZE), Image.NEAREST)) > 127
    rt = np.asarray(Image.open(r["right"]).convert("L").resize((SIZE, SIZE), Image.NEAREST)) > 127
    return (l | rt).astype(np.float32)
def dice_np(p, g):
    p, g = p >= 0.5, g >= 0.5; ps, gs = p.sum(), g.sum()
    return 1.0 if (ps == 0 and gs == 0) else float(2 * np.logical_and(p, g).sum() / (ps + gs))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("AttentionUNet params: %.2fM | device %s" % (sum(p.numel() for p in AttentionUNet().parameters())/1e6, DEVICE))


## 5. Train lung-field segmentation

In [ ]:
class SegDS(Dataset):
    def __init__(self, f, aug=False): self.f = f.reset_index(drop=True); self.aug = aug
    def __len__(self): return len(self.f)
    def __getitem__(self, i):
        r = self.f.iloc[i]; x = load_img(r["image"]); y = lung_mask(r)
        if self.aug and np.random.rand() < 0.5: x = x[:, :, ::-1].copy(); y = y[:, ::-1].copy()
        return torch.from_numpy(x), torch.from_numpy(y[None])

torch.manual_seed(42); np.random.seed(42)
tr = seg_df.sample(frac=0.8, random_state=42); va = seg_df.drop(tr.index)
seg = AttentionUNet(3, 1, base=32).to(DEVICE); opt = torch.optim.Adam(seg.parameters(), 1e-3)
tr_dl = DataLoader(SegDS(tr, True), batch_size=8, shuffle=True, num_workers=2)
va_dl = DataLoader(SegDS(va), batch_size=8, num_workers=2)
EPOCHS = 40; best = -1.0
for e in range(1, EPOCHS + 1):
    seg.train()
    for x, y in tr_dl:
        x, y = x.to(DEVICE), y.to(DEVICE); opt.zero_grad()
        dice_bce_loss(seg(x), y).backward(); opt.step()
    seg.eval(); ds = []
    with torch.no_grad():
        for x, y in va_dl:
            p = torch.sigmoid(seg(x.to(DEVICE))).cpu().numpy()
            for pi, gi in zip(p, y.numpy()): ds.append(dice_np(pi[0], gi[0]))
    vd = float(np.mean(ds))
    if vd > best: best = vd; torch.save(seg.state_dict(), "unet_cxr.pt")
    if e == 1 or e % 5 == 0: print(f"epoch {e}/{EPOCHS}  lung val_Dice={vd:.4f}")
print("best lung-seg Dice:", round(best, 4))


## 6. Train normal vs. TB classifier

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.metrics import roc_auc_score

class ClsDS(Dataset):
    def __init__(self, f, aug=False): self.f = f.reset_index(drop=True); self.aug = aug
    def __len__(self): return len(self.f)
    def __getitem__(self, i):
        r = self.f.iloc[i]; x = load_img(r["image"])
        if self.aug and np.random.rand() < 0.5: x = x[:, :, ::-1].copy()
        return torch.from_numpy(x), torch.tensor([float(r["tb"])])

ctr = df.sample(frac=0.8, random_state=42); cva = df.drop(ctr.index)
clf = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
clf.classifier[1] = nn.Linear(clf.classifier[1].in_features, 1); clf = clf.to(DEVICE)
posw = torch.tensor([(ctr.tb == 0).sum() / max((ctr.tb == 1).sum(), 1)], dtype=torch.float32, device=DEVICE)
crit = nn.BCEWithLogitsLoss(pos_weight=posw); copt = torch.optim.Adam(clf.parameters(), 3e-4)
ctr_dl = DataLoader(ClsDS(ctr, True), batch_size=16, shuffle=True, num_workers=2)
cva_dl = DataLoader(ClsDS(cva), batch_size=16, num_workers=2)

def auc():
    clf.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, y in cva_dl:
            p = torch.sigmoid(clf(x.to(DEVICE))).cpu().numpy().ravel()
            ps += p.tolist(); ys += y.numpy().ravel().tolist()
    return roc_auc_score(ys, ps)

cbest = -1.0
for e in range(1, 9):
    clf.train()
    for x, y in ctr_dl:
        x, y = x.to(DEVICE), y.to(DEVICE); copt.zero_grad(); crit(clf(x), y).backward(); copt.step()
    a = auc()
    if a > cbest: cbest = a; torch.save(clf.state_dict(), "classifier_cxr.pt")
    print(f"epoch {e}/8  TB val_AUROC={a:.4f}")
print("best TB AUROC:", round(cbest, 4))


## 7. Export ONNX (lung-seg + TB classifier)

In [ ]:
seg.load_state_dict(torch.load("unet_cxr.pt")); seg.eval().cpu()
torch.onnx.export(seg, torch.randn(1, 3, 256, 256), "unet_cxr.onnx",
                  input_names=["input"], output_names=["logits"],
                  dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17, dynamo=False)
clf.load_state_dict(torch.load("classifier_cxr.pt")); clf.eval().cpu()
torch.onnx.export(clf, torch.randn(1, 3, 256, 256), "classifier_cxr.onnx",
                  input_names=["input"], output_names=["logit"],
                  dynamic_axes={"input": {0: "batch"}}, opset_version=17, dynamo=False)
import onnxruntime as ort
for f in ["unet_cxr.onnx", "classifier_cxr.onnx"]:
    ort.InferenceSession(f, providers=["CPUExecutionProvider"]); print("OK", f)
from google.colab import files
files.download("unet_cxr.onnx"); files.download("classifier_cxr.onnx")


## 8. Build the CXR case bundle (image + lung mask + normal/TB label)

In [ ]:
import csv, zipfile, shutil
OUT = "bundle_cxr"
if os.path.isdir(OUT): shutil.rmtree(OUT)
os.makedirs(f"{OUT}/images"); os.makedirs(f"{OUT}/gt")
rows = []
for _, r in seg_df.head(100).iterrows():
    cid = r["id"]
    Image.open(r["image"]).convert("RGB").resize((256, 256)).save(f"{OUT}/images/{cid}.png")
    Image.fromarray((lung_mask(r) * 255).astype("uint8")).save(f"{OUT}/gt/{cid}.png")
    rows.append({"case_id": cid, "diagnosis": "tb" if r["tb"] == 1 else "normal"})
with open(f"{OUT}/labels.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["case_id", "diagnosis"]); w.writeheader(); w.writerows(rows)
with zipfile.ZipFile("case_bundle_cxr.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(OUT):
        for fn in fs:
            p = os.path.join(root, fn); z.write(p, os.path.relpath(p, "."))
print(f"exported {len(rows)} CXR cases ({sum(r['diagnosis']=='tb' for r in rows)} TB) -> case_bundle_cxr.zip")
files.download("case_bundle_cxr.zip")


## Done — three artifacts to bring back
- `unet_cxr.onnx` → `ml/models/unet_cxr.onnx`  (lung-field segmentation)
- `classifier_cxr.onnx` → `ml/models/classifier_cxr.onnx`  (normal vs TB)
- `case_bundle_cxr.zip` → the ~100 chest-X-ray play cases (image + lung mask + label)

The multi-modality backend is **ready**. Put the two ONNX files in `ml/models/`, then:

```bash
cd web/backend && source .venv/bin/activate
python load_bundle.py ~/Downloads/case_bundle_cxr.zip
python seed_cases.py --modality chest_xray --bundle ../data/bundle/bundle
python app.py
```

Then pick **Chest X-ray** in the Modality dropdown to play it.

**Caveats:** lung masks come from Montgomery (~138) so the seg set is small (lung fields are large/simple, so
Dice is still high); the TB classifier uses an image-level split; research/education, not a diagnostic device.